In [ ]:
# ============================================================
# Colab code: Add GN division + GN population to crime dataset
# ============================================================

!pip -q install scikit-learn

import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

# 2) EDIT THESE PATHS (point to where your files are in Colab)
CRIME_CSV_PATH = "/content/crime_data_final.csv"
GN_CSV_PATH    = "/content/kandy_gn_divisions.csv"
OUTPUT_PATH    = "/content/crime_data_final_with_gn_and_population.csv"

# 3) LOAD DATA

crime = pd.read_csv(CRIME_CSV_PATH)
gn = pd.read_csv(GN_CSV_PATH)

# 4) COORDINATE CLEANUP
crime["Latitude"]  = pd.to_numeric(crime["Latitude"], errors="coerce")
crime["Longitude"] = pd.to_numeric(crime["Longitude"], errors="coerce")

gn["latitude"]  = pd.to_numeric(gn["latitude"], errors="coerce")
gn["longitude"] = pd.to_numeric(gn["longitude"], errors="coerce")

if crime[["Latitude", "Longitude"]].isna().any().any():
    bad = crime[crime["Latitude"].isna() | crime["Longitude"].isna()]
    raise ValueError(f"Crime CSV has missing coordinates. Sample rows:\n{bad.head(5)}")

if gn[["latitude", "longitude"]].isna().any().any():
    bad = gn[gn["latitude"].isna() | gn["longitude"].isna()]
    raise ValueError(f"GN CSV has missing coordinates. Sample rows:\n{bad.head(5)}")

# 5) NEAREST-GN MATCHING (HAVERSINE)
crime_rad = np.deg2rad(crime[["Latitude", "Longitude"]].to_numpy())
gn_rad    = np.deg2rad(gn[["latitude", "longitude"]].to_numpy())

tree = BallTree(gn_rad, metric="haversine")
dist_rad, idx = tree.query(crime_rad, k=1)

idx = idx.flatten()
dist_rad = dist_rad.flatten()

# 6) ATTACH GN COLUMNS (division + pcode + population)
crime["GN division"]    = gn.iloc[idx]["admin4Name_en"].to_numpy()
crime["GN_pcode"]       = gn.iloc[idx]["admin4Pcode"].to_numpy()
crime["GN_population"]  = gn.iloc[idx]["population"].to_numpy()

EARTH_RADIUS_M = 6_371_000
crime["GN_distance_m"] = (dist_rad * EARTH_RADIUS_M).round(1)

# 7) SAVE
crime.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")

crime.head()


Total rows in Kandy CSV: 1187
Rows missing coordinates: 0

Saved output: /content/kandy_gn_divisions_with_coords.csv
